In [ ]:
!pip install autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
datasetPath = Path("/content/drive/MyDrive/ML-For-CV-Robustness/datasets/datasetGTnoTrees.csv")

#making sure
datasetPath.is_file()

True

In [ ]:
dataset = pd.read_csv(datasetPath)
dataset

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333475,0.262388,0.000241,0.000365,550.956812,480.270977,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333476,0.239909,0.000271,0.000399,550.956812,480.270977,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333477,0.284332,0.000270,0.000425,550.956812,480.270977,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333478,0.241989,0.000346,0.000511,550.956812,480.270977,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [ ]:
from autogluon.tabular import TabularPredictor

In [ ]:
train_df = dataset.sample(frac=0.8, random_state=42)
test_df = dataset.drop(train_df.index)

In [ ]:
train_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
142331,0.145998,0.000409,0.000513,531.714557,479.878278,11.0,180.0,-63.688171,56.363342,0.0,-19.532,-0.000,0.30
270207,0.311861,0.000227,0.000388,368.744438,321.424198,7.0,180.0,-42.151062,85.965363,0.0,-19.320,-0.000,0.38
49351,0.270059,0.000188,0.000296,220.893233,200.472881,11.0,180.0,-71.945084,19.489212,0.0,-22.271,180.000,0.34
169549,0.229224,0.000165,0.000216,1130.973355,1020.232214,9.0,180.0,-63.812729,-56.074120,-0.0,-22.112,-180.000,0.34
31301,0.304657,0.000192,0.000316,1334.391480,1020.232214,1.0,180.0,-19.763107,-104.398338,0.0,-22.510,180.000,0.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...
304611,0.233003,0.000227,0.000312,531.714557,479.878278,11.0,180.0,-63.688171,56.363342,-0.0,-21.426,-180.000,0.28
119616,0.270809,0.000170,0.000217,370.118885,322.798645,16.0,180.0,-42.301559,-85.835487,0.0,-19.813,81.537,0.20
59809,0.250882,0.000259,0.000396,228.550866,199.294784,9.0,180.0,-42.151062,85.965363,0.0,-18.992,-0.000,0.34
273354,0.222158,0.000413,0.000600,368.744438,321.424198,14.0,180.0,-42.151062,85.965363,-0.0,-18.932,-0.000,0.24


In [ ]:
test_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
5,0.081775,0.000463,0.000538,1130.973355,1020.232214,5.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
13,0.108716,0.000371,0.000444,1130.973355,1020.232214,13.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
15,0.144419,0.000328,0.000411,1130.973355,1020.232214,15.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
18,0.167532,0.000247,0.000317,1130.973355,1020.232214,18.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
19,0.185488,0.000243,0.000321,1130.973355,1020.232214,19.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333458,0.296286,0.000274,0.000436,550.956812,480.270977,18.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.26
333460,0.267418,0.000102,0.000135,550.956812,480.270977,0.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333470,0.239545,0.000260,0.000367,550.956812,480.270977,10.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333473,0.259494,0.000291,0.000437,550.956812,480.270977,13.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [ ]:
predictor = TabularPredictor(
    label="grviOut",
    problem_type="regression",
    eval_metric="root_mean_squared_error"
).fit(
    train_data=train_df
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260825_104222"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.79 GB / 12.67 GB (85.2%)
Disk Space Avail:   65.08 GB / 112.64 GB (57.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and t

[1000]	valid_set's rmse: 0.0338313
[2000]	valid_set's rmse: 0.0310242
[3000]	valid_set's rmse: 0.029477
[4000]	valid_set's rmse: 0.0284801
[5000]	valid_set's rmse: 0.0277691
[6000]	valid_set's rmse: 0.0271868
[7000]	valid_set's rmse: 0.0267275
[8000]	valid_set's rmse: 0.0262835
[9000]	valid_set's rmse: 0.0259175
[10000]	valid_set's rmse: 0.0256152


	-0.0256	 = Validation score   (-root_mean_squared_error)
	166.54s	 = Training   runtime
	2.73s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=1, gpus=0, mem=0.2/10.6 GB


[1000]	valid_set's rmse: 0.0262214
[2000]	valid_set's rmse: 0.0235341
[3000]	valid_set's rmse: 0.0223233
[4000]	valid_set's rmse: 0.0215914
[5000]	valid_set's rmse: 0.0211341
[6000]	valid_set's rmse: 0.0207258
[7000]	valid_set's rmse: 0.0204706
[8000]	valid_set's rmse: 0.0202143
[9000]	valid_set's rmse: 0.0200095
[10000]	valid_set's rmse: 0.0198544


	-0.0199	 = Validation score   (-root_mean_squared_error)
	112.44s	 = Training   runtime
	1.91s	 = Validation runtime
Fitting model: RandomForestMSE ...
	Fitting with cpus=2, gpus=0, mem=1.3/10.6 GB
	-0.0239	 = Validation score   (-root_mean_squared_error)
	543.99s	 = Training   runtime
	0.27s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=1, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=2, gpus=0, mem=1.3/10.6 GB
	-0.0186	 = Validation score   (-root_mean_squared_error)
	92.89s	 = Training   runtime
	0.4s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=1, gpus=0, mem=0.2/10.4 GB
	-0.0306	 = Validation score   (-root_mean_squared_error)
	167.61s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: XGBoost ...
	Fitting with cpus=1, gpus=0
	-0.019	 = Validation score   (-root_mean_squared_error)
	236.74s	 =

[1000]	valid_set's rmse: 0.0229421
[2000]	valid_set's rmse: 0.0212138
[3000]	valid_set's rmse: 0.0205481
[4000]	valid_set's rmse: 0.0201424
[5000]	valid_set's rmse: 0.0199193
[6000]	valid_set's rmse: 0.0197567
[7000]	valid_set's rmse: 0.0196243
[8000]	valid_set's rmse: 0.0194582
[9000]	valid_set's rmse: 0.0193621
[10000]	valid_set's rmse: 0.0193071


	-0.0193	 = Validation score   (-root_mean_squared_error)
	133.25s	 = Training   runtime
	2.61s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.2 GB
	Ensemble Weights: {'ExtraTreesMSE': 0.417, 'XGBoost': 0.333, 'LightGBMLarge': 0.208, 'NeuralNetTorch': 0.042}
	-0.017	 = Validation score   (-root_mean_squared_error)
	0.03s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 6681.57s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 718.7 rows/s (2668 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20260825_104222")


In [ ]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.016959,root_mean_squared_error,3.712499,5658.570760,0.000770,0.029971,2,True,9
1,ExtraTreesMSE,-0.018576,root_mean_squared_error,0.401180,92.891487,0.401180,92.891487,1,True,4
2,XGBoost,-0.018998,root_mean_squared_error,0.683534,236.743962,0.683534,236.743962,1,True,6
3,LightGBMLarge,-0.019307,root_mean_squared_error,2.608999,133.253639,2.608999,133.253639,1,True,8
4,LightGBM,-0.019854,root_mean_squared_error,1.909771,112.443420,1.909771,112.443420,1,True,2
5,RandomForestMSE,-0.023900,root_mean_squared_error,0.265890,543.990200,0.265890,543.990200,1,True,3
6,LightGBMXT,-0.025615,root_mean_squared_error,2.730135,166.544299,2.730135,166.544299,1,True,1
7,NeuralNetTorch,-0.026338,root_mean_squared_error,0.018017,5195.651701,0.018017,5195.651701,1,True,7
8,NeuralNetFastAI,-0.030642,root_mean_squared_error,0.032948,167.605137,0.032948,167.605137,1,True,5


In [ ]:
predictor.leaderboard(test_df, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.016611,-0.016959,root_mean_squared_error,88.942528,3.712499,5658.570760,0.020288,0.000770,0.029971,2,True,9
1,XGBoost,-0.018421,-0.018998,root_mean_squared_error,18.452009,0.683534,236.743962,18.452009,0.683534,236.743962,1,True,6
2,ExtraTreesMSE,-0.018534,-0.018576,root_mean_squared_error,4.111614,0.401180,92.891487,4.111614,0.401180,92.891487,1,True,4
3,LightGBMLarge,-0.018878,-0.019307,root_mean_squared_error,66.077920,2.608999,133.253639,66.077920,2.608999,133.253639,1,True,8
4,LightGBM,-0.019776,-0.019854,root_mean_squared_error,46.788990,1.909771,112.443420,46.788990,1.909771,112.443420,1,True,2
5,RandomForestMSE,-0.024028,-0.023900,root_mean_squared_error,2.983717,0.265890,543.990200,2.983717,0.265890,543.990200,1,True,3
6,LightGBMXT,-0.026168,-0.025615,root_mean_squared_error,68.136531,2.730135,166.544299,68.136531,2.730135,166.544299,1,True,1
7,NeuralNetTorch,-0.027537,-0.026338,root_mean_squared_error,0.280696,0.018017,5195.651701,0.280696,0.018017,5195.651701,1,True,7
8,NeuralNetFastAI,-0.031516,-0.030642,root_mean_squared_error,0.666484,0.032948,167.605137,0.666484,0.032948,167.605137,1,True,5
